In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Conv2D, MaxPooling2D, Flatten
import matplotlib.pyplot as plt

In [ ]:
# Set your dataset directory
directory = 'C:/Data/Codes/Datasets/PetImages/'

In [ ]:
# Function to check image validity
def check_image_validity(filepath):
    try:
        image = cv2.imread(filepath)
        if image is not None and image.size > 0:
            # Check for 1, 3, or 4 color channels
            if image.shape[-1] in [1, 3, 4]:
                return True
            else:
                print(f"Invalid number of channels in {filepath}: {image.shape[-1]} channels.")
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
    return False

In [ ]:
# Function to filter valid images
def filter_valid_images(base_directory):
    valid_images = []
    for category in ['Cat/', 'Dog/']:  # Adjust as necessary
        category_dir = os.path.join(base_directory, category)
        for filename in os.listdir(category_dir):
            filepath = os.path.join(category_dir, filename)
            print(f"Checking {filepath}...")
            if check_image_validity(filepath):
                valid_images.append(filepath)
    return valid_images

In [ ]:
# Filter valid training images (assuming they are in the same folder; can be adjusted based on your structure)
valid_images = filter_valid_images(directory)

In [ ]:
# Create datasets from valid images
train_images, train_labels = create_image_dataset(valid_images)

# Convert to TensorFlow dataset
train_ds = tf.data.Dataset.from_tensor_slices((train_images, train_labels))

In [ ]:
# Normalize function
def process(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

In [ ]:
# Map normalization
train_ds = train_ds.map(process)

In [ ]:
# Shuffle and batch the datasets
batch_size = 32
train_ds = train_ds.shuffle(1000).batch(batch_size)

In [ ]:
# Enable eager execution for debugging
tf.config.run_functions_eagerly(True)

In [ ]:
# CNN Model Building
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3), padding='valid', activation='relu', input_shape=(256, 256, 3)))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

model.add(Conv2D(64, kernel_size=(3, 3), padding='valid', activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

model.add(Conv2D(128, kernel_size=(3, 3), padding='valid', activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2), strides=2, padding='valid'))

model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))  # Use 'softmax' for multi-class

In [ ]:
# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Model summary
model.summary()

# Fit the model
history = model.fit(train_ds, epochs=10)

In [ ]:
# Plotting accuracy
plt.plot(history.history['accuracy'], color='red', label='Train')
plt.legend()
plt.show()

In [ ]:
model.save('dmicucnn.h5')